# LAB | Relevance Scoring and Rerankers for Trustworthy AI & EU AI Act

## Learning Objectives
- Build a complete RAG pipeline with metadata-aware chunking over heterogeneous documents
- Compare baseline vector retrieval with advanced reranking strategies
- Evaluate retrieval performance with Precision@5, MRR, and NDCG@5
- Understand when reranking adds value and when metadata filtering is sufficient

## Pipeline Architecture
```
DATA → CHUNKING → EMBEDDING → VECTOR DB → QUERY EMBEDDING → RETRIEVAL → RERANKING → GENERATION
```

## Core vs Advanced Features

| Feature | Level |
|---------|-------|
| Chunking with metadata | Core |
| OpenAI embeddings + Pinecone | Core |
| Baseline retrieval | Core |
| Metadata filtering | Core |
| Query enhancement (rewriting, expansion, step-back) | Core |
| Evaluation: Precision@5, MRR, NDCG@5 | Core |
| LLM-based relevance scoring | **Advanced** |
| Cohere Rerank | **Advanced** |
| CrossEncoder (local) | **Advanced** |

---
## Section 0 — Setup and Environment

In [ ]:
import os
import json
import time
import warnings
import importlib.util
from pathlib import Path
from typing import List, Dict, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

try:
    from pinecone import Pinecone, ServerlessSpec
except ImportError:
    Pinecone = None
    ServerlessSpec = None

warnings.filterwarnings('ignore')
load_dotenv(dotenv_path=Path('.env'), override=False)

# Constants
OPENAI_MODEL   = 'gpt-4o-mini'
EMBED_MODEL    = 'text-embedding-3-small'
EMBED_DIM      = 1536
PINECONE_INDEX = os.getenv('PINECONE_INDEX_NAME', 'relevance-scoring-lab')
INITIAL_TOP_K  = 12
FINAL_TOP_K    = 5
FORCE_REINDEX  = False  # set True to re-embed and re-upload all chunks

OPENAI_API_KEY   = os.getenv('OPENAI_API_KEY')
PINECONE_API_KEY = os.getenv('PINECONE_API_KEY')
COHERE_API_KEY   = os.getenv('COHERE_API_KEY')

# Clients are optional so the notebook can still run and explain missing setup.
oai = OpenAI(api_key=OPENAI_API_KEY) if OpenAI and OPENAI_API_KEY else None
pc  = Pinecone(api_key=PINECONE_API_KEY) if Pinecone and PINECONE_API_KEY else None
index = None

HAS_OPENAI   = oai is not None
HAS_PINECONE = pc is not None
HAS_COHERE   = bool(COHERE_API_KEY) and importlib.util.find_spec('cohere') is not None


def pinecone_eq(field: str, value) -> Dict:
    return {field: {'$eq': value}}


print('Environment loaded')
print(f'  OpenAI model  : {OPENAI_MODEL}')
print(f'  Embed model   : {EMBED_MODEL} ({EMBED_DIM} dimensions)')
print(f'  Pinecone index: {PINECONE_INDEX}')
print(f'  Initial top-k : {INITIAL_TOP_K}  |  Final top-k: {FINAL_TOP_K}')
print('\nAPI/package status:')
print(f'  OpenAI   : {"ready" if HAS_OPENAI else "missing package or OPENAI_API_KEY"}')
print(f'  Pinecone : {"ready" if HAS_PINECONE else "missing package or PINECONE_API_KEY"}')
print(f'  Cohere   : {"ready" if HAS_COHERE else "missing package or COHERE_API_KEY"}')


---
## Section 1 — Data Loading and Inspection

Update `DOC_REGISTRY` with the actual filenames your professor provided.
The `category` field maps to our metadata schema and drives metadata filtering later.

In [ ]:
# Professor-provided files — actual filenames from data/
# Add a third PDF to this list if your professor provides one.
DOC_REGISTRY = [
    {
        'path'     : 'data/pdfs/eu_ai_act.pdf',
        'doc_id'   : 'eu_ai_act',
        'doc_title': 'EU AI Act',
        'category' : 'regulation',
        'type'     : 'pdf',
    },
    {
        # HLEG Ethics Guidelines for Trustworthy AI (English)
        'path'     : 'data/pdfs/ai_hleg_ethics_guidelines_for_trustworthy_ai-en_87F84A41-A6E8-F38C-BFF661481B40077B_60419.pdf',
        'doc_id'   : 'hleg_guidelines_en',
        'doc_title': 'HLEG Ethics Guidelines for Trustworthy AI (EN)',
        'category' : 'class_material',
        'type'     : 'pdf',
    },
    {
        # HLEG Ethics Guidelines for Trustworthy AI (French)
        # Keep for multilingual demo or remove if you only need English content.
        'path'     : 'data/pdfs/ethics_guidelines_for_trustworthy_ai-fr_87FE7A3C-D03D-9305-81A653DDA84B5A60_60427.pdf',
        'doc_id'   : 'hleg_guidelines_fr',
        'doc_title': 'HLEG Ethics Guidelines for Trustworthy AI (FR)',
        'category' : 'class_material',
        'type'     : 'pdf',
    },
    {
        # Podcast transcript — generated by Section 1.3 if not yet present
        'path'     : 'data/transcripts/trustworthy_ai_podcast.txt',
        'doc_id'   : 'trustworthy_ai_podcast',
        'doc_title': 'The Blueprint for Trustworthy AI (Podcast)',
        'category' : 'podcast',
        'type'     : 'transcript',
    },
]

In [ ]:
from pypdf import PdfReader

print('File inventory:')
for doc in DOC_REGISTRY:
    p = Path(doc['path'])
    if not p.exists():
        print(f'  MISSING  {doc["path"]}')
        continue
    if doc['type'] == 'pdf':
        reader = PdfReader(doc['path'])
        print(f'  OK  {doc["doc_id"]:30s}  {len(reader.pages):4d} pages  [{doc["category"]}]')
    else:
        chars = len(p.read_text(encoding='utf-8'))
        print(f'  OK  {doc["doc_id"]:30s}  {chars:7,} chars  [{doc["category"]}]')

### 1.3 — Audio Transcription (optional)

Run the cell below only if you have an audio file and no transcript yet.
Transcription uses OpenAI Whisper and saves to `data/transcripts/`.

In [ ]:
AUDIO_PATH     = 'data/Audio_Podcast/The_Blueprint_For_Trustworthy_AI.m4a'
TRANSCRIPT_OUT = 'data/transcripts/trustworthy_ai_podcast.txt'

if Path(AUDIO_PATH).exists() and not Path(TRANSCRIPT_OUT).exists():
    if not HAS_OPENAI:
        print(f'Audio exists at {AUDIO_PATH}, but OpenAI is not configured; skipping transcription.')
    else:
        print(f'Transcribing {AUDIO_PATH} — this may take a minute ...')
        with open(AUDIO_PATH, 'rb') as f:
            transcript = oai.audio.transcriptions.create(model='whisper-1', file=f)
        Path(TRANSCRIPT_OUT).parent.mkdir(parents=True, exist_ok=True)
        Path(TRANSCRIPT_OUT).write_text(transcript.text, encoding='utf-8')
        print(f'Saved transcript to {TRANSCRIPT_OUT}')
elif Path(TRANSCRIPT_OUT).exists():
    print(f'Transcript already exists: {TRANSCRIPT_OUT}')
    print(f'  Size: {len(Path(TRANSCRIPT_OUT).read_text()):,} chars')
else:
    print(f'Audio file not found at {AUDIO_PATH}; transcript document will be skipped.')


---
## Section 2 — Chunking and Metadata Tagging

Each chunk carries a flat metadata dict that Pinecone can filter on.

**Metadata schema for legal documents:**
```python
{
    'source_type': 'legal_document',   # legal_document | pdf | podcast_transcript
    'doc_id'     : 'eu_ai_act',
    'doc_title'  : 'EU AI Act',
    'category'   : 'regulation',       # regulation | class_material | podcast
    'page'       : 12,
    'section'    : '',
    'chunk_index': 3,
    'chunk_id'   : 'eu_ai_act_p12_c3',
    'parent_id'  : 'eu_ai_act_p12',
    'char_count' : 412,
    'language'   : 'en',
}
```
Podcast chunks use `source_type='podcast_transcript'` and `category='podcast'`.

**Pinecone-filterable fields:** `source_type`, `category`, `doc_id`, `language`

In [ ]:
from helpers.chunker import chunk_pdf, chunk_transcript

all_chunks: List[Dict] = []

for doc in DOC_REGISTRY:
    p = Path(doc['path'])
    if not p.exists():
        print(f'Skipping {doc["doc_id"]} — file not found')
        continue
    if doc['type'] == 'pdf':
        chunks = chunk_pdf(
            path=doc['path'],
            doc_id=doc['doc_id'],
            doc_title=doc['doc_title'],
            category=doc['category'],
        )
    else:
        chunks = chunk_transcript(
            path=doc['path'],
            doc_id=doc['doc_id'],
            doc_title=doc['doc_title'],
        )
    all_chunks.extend(chunks)
    print(f'  {doc["doc_id"]:30s}  {len(chunks):4d} chunks')

print(f'\nTotal chunks: {len(all_chunks)}')

In [ ]:
import random
random.seed(42)

for c in random.sample(all_chunks, min(3, len(all_chunks))):
    print('=' * 60)
    print(f'chunk_id : {c["chunk_id"]}')
    meta_display = {k: v for k, v in c['metadata'].items()}
    print(f'metadata : {json.dumps(meta_display, indent=2)}')
    print(f'text     : {c["text"][:200]}...')

---
## Section 3 — Embeddings and Pinecone Indexing

We embed with `text-embedding-3-small` (1536 dims) and upsert to Pinecone in batches of 100.
Chunk text is stored in the Pinecone metadata field `text` so it can be retrieved alongside scores.

In [ ]:
from helpers.embedder import DIMENSION

if DIMENSION != EMBED_DIM:
    raise ValueError(f'Embedding dimension mismatch: notebook={EMBED_DIM}, helper={DIMENSION}')

if not HAS_PINECONE:
    print('Skipping Pinecone index setup because Pinecone is not configured.')
    index = None
else:
    listed = pc.list_indexes()
    existing = listed.names() if hasattr(listed, 'names') else [getattr(idx, 'name', idx.get('name')) for idx in listed]
    if PINECONE_INDEX not in existing:
        print(f'Creating index "{PINECONE_INDEX}" ...')
        pc.create_index(
            name=PINECONE_INDEX,
            dimension=DIMENSION,
            metric='cosine',
            spec=ServerlessSpec(cloud='aws', region='us-east-1'),
        )
        print('Index created.')
    else:
        print(f'Index "{PINECONE_INDEX}" already exists.')

    index_desc = pc.describe_index(PINECONE_INDEX)
    index_dim = getattr(index_desc, 'dimension', None)
    if index_dim is None and isinstance(index_desc, dict):
        index_dim = index_desc.get('dimension')
    if index_dim != DIMENSION:
        raise ValueError(f'Pinecone index dimension is {index_dim}, expected {DIMENSION}. Create a matching index or change PINECONE_INDEX_NAME.')

    index = pc.Index(PINECONE_INDEX)
    print(index.describe_index_stats())


In [ ]:
from helpers.embedder import embed_texts

BATCH_SIZE = 100

def upsert_all_chunks(chunks: List[Dict], force: bool = False) -> None:
    if index is None:
        print('Skipping upsert because Pinecone is not configured.')
        return
    if not HAS_OPENAI:
        print('Skipping upsert because OpenAI embeddings are not configured.')
        return
    if not chunks:
        print('No chunks available to upsert. Add source PDFs/transcripts under data/ first.')
        return

    stats = index.describe_index_stats()
    if stats.total_vector_count > 0 and not force:
        print(f'Index already has {stats.total_vector_count} vectors.')
        print('Set FORCE_REINDEX = True to re-upload.')
        return

    print(f'Embedding and upserting {len(chunks)} chunks in batches of {BATCH_SIZE} ...')
    for i in tqdm(range(0, len(chunks), BATCH_SIZE)):
        batch = chunks[i:i + BATCH_SIZE]
        texts = [c['text'] for c in batch]
        embeddings = embed_texts(texts)
        vectors = [
            (c['chunk_id'], emb, {**c['metadata'], 'text': c['text']})
            for c, emb in zip(batch, embeddings)
        ]
        index.upsert(vectors=vectors)
    print(f'Done. Upserted {len(chunks)} vectors.')

upsert_all_chunks(all_chunks, force=FORCE_REINDEX)


In [ ]:
if index is None:
    print('Skipping index stats because Pinecone is not configured.')
else:
    stats = index.describe_index_stats()
    index_desc = pc.describe_index(PINECONE_INDEX)
    index_dim = getattr(index_desc, 'dimension', None)
    if index_dim is None and isinstance(index_desc, dict):
        index_dim = index_desc.get('dimension')
    print(f'Total vectors : {stats.total_vector_count}')
    print(f'Dimension     : {index_dim}')


---
## Section 4 — Baseline Retrieval

Baseline: embed query → retrieve top-`INITIAL_TOP_K` by cosine similarity → return top-`FINAL_TOP_K`.
No reranking, no filtering, no query enhancement. This is the reference we will compare against.

In [ ]:
from helpers.embedder import embed_query
from helpers.retriever import retrieve

TEST_QUERIES = [
    'What AI practices are prohibited under the EU AI Act?',
    'What are the key principles of trustworthy AI?',
    'How does the EU AI Act define high-risk AI systems?',
]

def show_results(results: List[Dict], title: str = 'Results') -> None:
    print(f'\n{"=" * 65}')
    print(f'  {title}')
    print(f'{"=" * 65}')
    for i, r in enumerate(results, 1):
        doc  = r['metadata'].get('doc_id', '?')
        page = r['metadata'].get('page', '-')
        print(f'  [{i}] score={r["score"]:.4f}  doc={doc}  page={page}')
        print(f'       {r["text"][:180]}...')

In [ ]:
query = TEST_QUERIES[0]
print(f'Query: {query}')

baseline_results = []
baseline_latency = 0.0

if index is None or not HAS_OPENAI:
    print('Skipping baseline retrieval because OpenAI and Pinecone are both required.')
else:
    t0 = time.time()
    query_emb        = embed_query(query)
    baseline_results = retrieve(query_emb, top_k=INITIAL_TOP_K)
    baseline_latency = time.time() - t0

    show_results(baseline_results[:FINAL_TOP_K], f'Baseline — top {FINAL_TOP_K}')
    print(f'\nLatency: {baseline_latency:.2f}s')


In [ ]:
# Run baseline on all test queries
if index is None or not HAS_OPENAI:
    print('Skipping baseline retrieval loop because OpenAI and Pinecone are both required.')
else:
    for q in TEST_QUERIES:
        qe  = embed_query(q)
        res = retrieve(qe, top_k=FINAL_TOP_K)
        show_results(res, q[:60])


---
## Section 5 — Metadata Filtering

Pinecone supports server-side metadata filtering.
Filtering before retrieval narrows the candidate set to semantically appropriate documents —
for example, restricting a legal question to `category='regulation'` removes podcast noise.

**Filterable fields:** `source_type`, `category`, `doc_id`, `language`

In [ ]:
query = TEST_QUERIES[0]

if index is None or not HAS_OPENAI:
    reg_results = []
    print('Skipping filtered retrieval because OpenAI and Pinecone are both required.')
else:
    qe = embed_query(query)
    reg_results = retrieve(qe, top_k=INITIAL_TOP_K, metadata_filter=pinecone_eq('category', 'regulation'))
    show_results(reg_results[:FINAL_TOP_K], 'Filtered: category=regulation')


In [ ]:
query = TEST_QUERIES[1]

if index is None or not HAS_OPENAI:
    pod_results = []
    print('Skipping filtered retrieval because OpenAI and Pinecone are both required.')
else:
    qe = embed_query(query)
    pod_results = retrieve(qe, top_k=INITIAL_TOP_K, metadata_filter=pinecone_eq('source_type', 'podcast_transcript'))
    show_results(pod_results[:FINAL_TOP_K], 'Filtered: source_type=podcast_transcript')


In [ ]:
query = TEST_QUERIES[2]

if index is None or not HAS_OPENAI:
    doc_results = []
    print('Skipping filtered retrieval because OpenAI and Pinecone are both required.')
else:
    qe = embed_query(query)
    doc_results = retrieve(qe, top_k=INITIAL_TOP_K, metadata_filter=pinecone_eq('doc_id', 'eu_ai_act'))
    show_results(doc_results[:FINAL_TOP_K], 'Filtered: doc_id=eu_ai_act')

    # Compare unfiltered vs filtered doc count
    unfiltered  = retrieve(qe, top_k=FINAL_TOP_K)
    docs_in_un  = [r['metadata'].get('doc_id') for r in unfiltered]
    docs_in_fil = [r['metadata'].get('doc_id') for r in doc_results[:FINAL_TOP_K]]
    print(f'\nUnfiltered doc_ids : {docs_in_un}')
    print(f'Filtered doc_ids   : {docs_in_fil}')


---
## Section 6 — Query Enhancement

Four strategies to improve what we send to the vector DB:

| Strategy | What it does |
|----------|--------------|
| **Query rewriting** | Rephrase for better semantic match to document language |
| **Sub-query decomposition** | Split compound questions into atomic sub-queries |
| **Step-back prompting** | Generalize to a parent concept to retrieve broader context |
| **Query expansion** | Add synonyms and related legal/AI terminology |

In [ ]:
def rewrite_query(query: str) -> str:
    if not HAS_OPENAI:
        print('OpenAI is not configured; using the original query.')
        return query
    prompt = (
        'Rewrite the following search query to improve retrieval from a legal and AI policy corpus.\n'
        'Make it more precise and use relevant legal/technical terminology.\n\n'
        f'Original query: {query}\n\n'
        'Rewritten query (one sentence, no explanation):'
    )
    resp = oai.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0, max_tokens=100,
    )
    return resp.choices[0].message.content.strip()


def decompose_query(query: str) -> List[str]:
    if not HAS_OPENAI:
        print('OpenAI is not configured; using the original query as the only sub-query.')
        return [query]
    prompt = (
        'Break the following query into 2-3 simpler sub-queries that together cover the full question.\n'
        'Return one sub-query per line with no numbering or bullet points.\n\n'
        f'Query: {query}'
    )
    resp = oai.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0, max_tokens=150,
    )
    return [ln.strip() for ln in resp.choices[0].message.content.strip().split('\n') if ln.strip()]


def step_back_query(query: str) -> str:
    if not HAS_OPENAI:
        print('OpenAI is not configured; using the original query.')
        return query
    prompt = (
        'Given the specific question below, write a broader, more general question that would '
        'help retrieve useful background context.\n\n'
        f'Specific question: {query}\n\n'
        'General step-back question (one sentence):'
    )
    resp = oai.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0, max_tokens=80,
    )
    return resp.choices[0].message.content.strip()


def expand_query(query: str) -> str:
    if not HAS_OPENAI:
        print('OpenAI is not configured; using the original query.')
        return query
    prompt = (
        'Expand the query by adding synonyms, related legal terms, and alternative phrasings. '
        'Return a single expanded query string.\n\n'
        f'Query: {query}\n\n'
        'Expanded query:'
    )
    resp = oai.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0, max_tokens=120,
    )
    return resp.choices[0].message.content.strip()


In [ ]:
original   = TEST_QUERIES[0]
rewritten  = rewrite_query(original)
stepback   = step_back_query(original)
expanded   = expand_query(original)
sub_qs     = decompose_query(original)

print(f'Original   : {original}')
print(f'Rewritten  : {rewritten}')
print(f'Step-back  : {stepback}')
print(f'Expanded   : {expanded}')
print(f'Sub-queries:')
for sq in sub_qs:
    print(f'  {sq}')

In [ ]:
# Compare retrieval: original vs rewritten query
if index is None or not HAS_OPENAI:
    orig_res = []
    rew_res = []
    print('Skipping retrieval comparison because OpenAI and Pinecone are both required.')
else:
    print('--- Original query ---')
    orig_res = retrieve(embed_query(original), top_k=FINAL_TOP_K)
    for r in orig_res:
        print(f'  [{r["score"]:.4f}] {r["metadata"].get("doc_id")} | {r["text"][:100]}...')

    print('\n--- Rewritten query ---')
    rew_res = retrieve(embed_query(rewritten), top_k=FINAL_TOP_K)
    for r in rew_res:
        print(f'  [{r["score"]:.4f}] {r["metadata"].get("doc_id")} | {r["text"][:100]}...')

    # Overlap analysis
    orig_ids = {r['chunk_id'] for r in orig_res}
    rew_ids  = {r['chunk_id'] for r in rew_res}
    print(f'\nChunk overlap: {len(orig_ids & rew_ids)} / {FINAL_TOP_K}')


In [ ]:
# Sub-query decomposition: merge and deduplicate results
seen_ids: set = set()
merged: List[Dict] = []

if index is None or not HAS_OPENAI:
    print('Skipping sub-query retrieval because OpenAI and Pinecone are both required.')
else:
    for sq in sub_qs:
        sq_results = retrieve(embed_query(sq), top_k=FINAL_TOP_K)
        for r in sq_results:
            if r['chunk_id'] not in seen_ids:
                merged.append(r)
                seen_ids.add(r['chunk_id'])

    print(f'Sub-query decomposition retrieved {len(merged)} unique chunks across {len(sub_qs)} sub-queries:')
    for r in merged[:FINAL_TOP_K]:
        print(f'  [{r["score"]:.4f}] {r["metadata"].get("doc_id")} | {r["text"][:100]}...')


---
## Section 7 — LLM-Based Relevance Scoring *(Advanced)*

Ask `gpt-4o-mini` to score each candidate 0–10 for relevance to the query.
Rerank by LLM score instead of cosine similarity.

**Trade-off:** flexible and model-agnostic, but slow (~12 API calls per query) and costs tokens.

In [ ]:
from helpers.reranker import llm_score_relevance

query      = TEST_QUERIES[0]
candidates = []
llm_results = []
llm_latency = 0.0

if index is None or not HAS_OPENAI:
    print('Skipping LLM scoring because OpenAI and Pinecone are both required.')
else:
    candidates = retrieve(embed_query(query), top_k=INITIAL_TOP_K)

    print(f'Scoring {len(candidates)} candidates with LLM ...')
    t0          = time.time()
    llm_results = llm_score_relevance(query, candidates, oai, top_n=FINAL_TOP_K)
    llm_latency = time.time() - t0

    print(f'\nLLM-Scored top {FINAL_TOP_K}  (latency: {llm_latency:.2f}s):')
    for i, r in enumerate(llm_results, 1):
        print(f'  [{i}] LLM={r["llm_score"]:.1f}  cosine={r["score"]:.4f}  doc={r["metadata"].get("doc_id")}')
        print(f'       {r["text"][:160]}...')


In [ ]:
# Show rank changes: baseline vs LLM scoring
baseline_ids = [r['chunk_id'] for r in candidates[:FINAL_TOP_K]]
llm_ids      = [r['chunk_id'] for r in llm_results]

if not baseline_ids:
    print('Skipping rank-change summary because no candidates are available.')
else:
    print('Rank changes after LLM scoring:')
    print(f'  Baseline  : {baseline_ids}')
    print(f'  LLM-scored: {llm_ids}')
    moved = sum(1 for a, b in zip(baseline_ids, llm_ids) if a != b)
    print(f'  {moved} / {FINAL_TOP_K} positions changed')


---
## Section 8 — Cohere Reranking *(Advanced)*

Cohere Rerank is a cross-encoder model served as an API.
It scores each (query, document) pair and returns a `relevance_score`.

Pattern: broad vector retrieval (`top_k=12`) → Cohere rerank → return top 5.

**Trade-off:** fast API call, high quality, but external dependency and per-request cost.

In [ ]:
from helpers.reranker import cohere_rerank

query      = TEST_QUERIES[0]
candidates = []
cohere_results = []
cohere_latency = 0.0

if index is None or not HAS_OPENAI:
    print('Skipping Cohere reranking because OpenAI and Pinecone are both required for candidate retrieval.')
else:
    candidates = retrieve(embed_query(query), top_k=INITIAL_TOP_K)

    print(f'Reranking {len(candidates)} candidates with Cohere ...')
    t0              = time.time()
    cohere_results  = cohere_rerank(query, candidates, top_n=FINAL_TOP_K)
    cohere_latency  = time.time() - t0

    print(f'\nCohere Reranked top {FINAL_TOP_K}  (latency: {cohere_latency:.2f}s):')
    for i, r in enumerate(cohere_results, 1):
        rerank_score = r.get('rerank_score', r.get('score', 0.0))
        print(f'  [{i}] rerank={rerank_score:.4f}  cosine={r["score"]:.4f}  doc={r["metadata"].get("doc_id")}')
        print(f'       {r["text"][:160]}...')


In [ ]:
# Rank changes: baseline vs Cohere
baseline_ids = [r['chunk_id'] for r in candidates[:FINAL_TOP_K]]
cohere_ids   = [r['chunk_id'] for r in cohere_results]

if not baseline_ids:
    print('Skipping rank-change summary because no candidates are available.')
else:
    print('Rank changes after Cohere reranking:')
    print(f'  Baseline: {baseline_ids}')
    print(f'  Cohere  : {cohere_ids}')
    moved = sum(1 for a, b in zip(baseline_ids, cohere_ids) if a != b)
    print(f'  {moved} / {FINAL_TOP_K} positions changed')


---
## Section 9 — CrossEncoder Reranking *(Advanced — Optional)*

CrossEncoders score (query, document) pairs using a single forward pass.
This section uses `cross-encoder/ms-marco-MiniLM-L-6-v2` from `sentence-transformers`.

**Before running:** uncomment `sentence-transformers` and `torch` in `requirements.txt`, then:
```bash
pip install sentence-transformers torch
```
The model download is ~90 MB. If the import fails, the function returns baseline order with a warning.

In [ ]:
from helpers.reranker import crossencoder_rerank

query      = TEST_QUERIES[0]
candidates = []
ce_results = []
ce_latency = 0.0

if index is None or not HAS_OPENAI:
    print('Skipping CrossEncoder reranking because OpenAI and Pinecone are both required for candidate retrieval.')
else:
    candidates = retrieve(embed_query(query), top_k=INITIAL_TOP_K)

    print(f'Reranking {len(candidates)} candidates with CrossEncoder ...')
    t0          = time.time()
    ce_results  = crossencoder_rerank(query, candidates, top_n=FINAL_TOP_K)
    ce_latency  = time.time() - t0

    print(f'\nCrossEncoder top {FINAL_TOP_K}  (latency: {ce_latency:.2f}s):')
    for i, r in enumerate(ce_results, 1):
        score = r.get('crossencoder_score', r['score'])
        print(f'  [{i}] ce_score={score:.4f}  doc={r["metadata"].get("doc_id")}')
        print(f'       {r["text"][:160]}...')


---
## Section 10 — Full RAG Pipeline with Reranking

End-to-end function: query enhancement → retrieval → reranking → generation.
The `reranker` argument switches between pipeline variants.

In [ ]:
def rag_pipeline(
    query: str,
    reranker: str = 'cohere',           # 'none' | 'llm' | 'cohere' | 'crossencoder'
    metadata_filter: Optional[Dict] = None,
    enhance_query: bool = True,
) -> str:
    if index is None or not HAS_OPENAI:
        return 'RAG pipeline skipped because OpenAI and Pinecone are both required.'

    # 1. Query enhancement
    retrieval_query = rewrite_query(query) if enhance_query else query

    # 2. Embed + retrieve broad candidate set
    qe         = embed_query(retrieval_query)
    candidates = retrieve(qe, top_k=INITIAL_TOP_K, metadata_filter=metadata_filter)

    # 3. Rerank
    if reranker == 'cohere':
        top_chunks = cohere_rerank(query, candidates, top_n=FINAL_TOP_K)
    elif reranker == 'llm':
        top_chunks = llm_score_relevance(query, candidates, oai, top_n=FINAL_TOP_K)
    elif reranker == 'crossencoder':
        top_chunks = crossencoder_rerank(query, candidates, top_n=FINAL_TOP_K)
    else:
        top_chunks = candidates[:FINAL_TOP_K]

    # 4. Build context
    context = '\n\n---\n\n'.join(
        f'[{c["metadata"].get("doc_id")} | p.{c["metadata"].get("page", "-")}]\n{c["text"]}'
        for c in top_chunks
    )

    # 5. Generate
    system = (
        'You are an expert on EU AI policy and trustworthy AI. '
        'Answer the question using only the provided context. '
        'Cite the document and page when possible. '
        'If the context does not contain the answer, say so clearly.'
    )
    resp = oai.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user',   'content': f'Context:\n{context}\n\nQuestion: {query}'},
        ],
        temperature=0.2,
        max_tokens=500,
    )
    return resp.choices[0].message.content


In [ ]:
# Test the full pipeline with Cohere reranking
query  = 'What are the prohibited AI practices in the EU AI Act?'
answer = rag_pipeline(query, reranker='cohere', metadata_filter=pinecone_eq('category', 'regulation'))
print(f'Query: {query}\n')
print(answer)


In [ ]:
# Compare answers: baseline vs Cohere reranker
query = 'What transparency obligations apply to AI systems?'

print('=== BASELINE (no reranking) ===')
baseline_answer = rag_pipeline(query, reranker='none', enhance_query=False)
print(baseline_answer)

print('\n=== COHERE RERANKED ===')
cohere_answer = rag_pipeline(query, reranker='cohere')
print(cohere_answer)

---
## Section 11 — Manual Evaluation

### How to build ground truth

1. Run baseline retrieval for each query in `eval_queries.json`.
2. Review the top-12 results manually.
3. For each chunk, assign a relevance label:
   - `2` = highly relevant (directly answers the query)
   - `1` = somewhat relevant (related but not precise)
   - `0` = not relevant
4. Add the chunk IDs to `relevant_chunk_ids` (label ≥ 1) and populate `relevance_map`.
5. Re-run this section.

**Metrics computed:** Precision@5, MRR, NDCG@5

In [ ]:
from helpers.evaluator import load_eval_queries, evaluate_pipeline

eval_queries = load_eval_queries('eval_queries.json')

for eq in eval_queries:
    status = 'READY' if eq['relevant_chunk_ids'] else 'NEEDS LABELS'
    print(f'  [{status:12s}] {eq["query_id"]}: {eq["query"][:70]}')

labeled = sum(1 for eq in eval_queries if eq['relevant_chunk_ids'])
print(f'\n{labeled}/{len(eval_queries)} queries have labels.')
if labeled == 0:
    print('Run baseline retrieval below, review results, then update eval_queries.json.')

In [ ]:
# Run baseline retrieval for each eval query so you can label the results
print('Baseline results for manual labeling\n')
print('Copy relevant chunk_ids into eval_queries.json\n')
print('=' * 65)

if index is None or not HAS_OPENAI:
    print('Skipping manual-label retrieval because OpenAI and Pinecone are both required.')
else:
    for eq in eval_queries:
        f   = eq.get('filter') or None
        qe  = embed_query(eq['query'])
        res = retrieve(qe, top_k=INITIAL_TOP_K, metadata_filter=f)
        print(f'\nQuery [{eq["query_id"]}]: {eq["query"]}')
        for i, r in enumerate(res[:8], 1):
            print(f'  [{i:2d}] id={r["chunk_id"]}  score={r["score"]:.4f}  doc={r["metadata"].get("doc_id")}')
            print(f'        {r["text"][:120]}...')


In [ ]:
# Run all pipelines for every eval query
pipeline_results: Dict[str, List] = {
    'Baseline'      : [],
    'LLM Scoring'   : [],
    'Cohere Rerank' : [],
}

if index is None or not HAS_OPENAI:
    print('Skipping live evaluation because OpenAI and Pinecone are both required.')
    for _ in eval_queries:
        pipeline_results['Baseline'].append([])
        pipeline_results['LLM Scoring'].append([])
        pipeline_results['Cohere Rerank'].append([])
else:
    for eq in tqdm(eval_queries, desc='Evaluating queries'):
        q  = eq['query']
        f  = eq.get('filter') or None
        qe = embed_query(q)

        candidates = retrieve(qe, top_k=INITIAL_TOP_K, metadata_filter=f)

        pipeline_results['Baseline'].append(candidates[:FINAL_TOP_K])
        pipeline_results['LLM Scoring'].append(
            llm_score_relevance(q, candidates, oai, top_n=FINAL_TOP_K)
        )
        pipeline_results['Cohere Rerank'].append(
            cohere_rerank(q, candidates, top_n=FINAL_TOP_K)
        )

print('Done.')


In [ ]:
metrics = []
for name, results in pipeline_results.items():
    m = evaluate_pipeline(name, results, eval_queries, k=FINAL_TOP_K)
    metrics.append(m)

df_metrics = pd.DataFrame(metrics).set_index('pipeline')
print(df_metrics.to_string())

df_metrics.to_csv('outputs/eval_results.csv')
print('\nSaved to outputs/eval_results.csv')

---
## Section 12 — Results and Conclusion

### Analysis template — fill in after running Section 11

**When did reranking help most?**
- Which query types improved most after Cohere reranking?
- Did legal document queries or podcast queries benefit more?
- Did metadata filtering already solve some retrieval problems without needing reranking?

**Pipeline comparison:**

| Pipeline | Strength | Weakness |
|----------|----------|----------|
| Baseline | Fastest, no extra cost | Lower precision on ambiguous queries |
| LLM Scoring | Most flexible, no separate rerank API | Slow (N × 1 LLM calls), expensive |
| Cohere Rerank | Good precision, fast API | External dependency, per-call cost |
| CrossEncoder | Local, no API cost | Model download, slower on CPU |

**Recommendation:**
Reranking adds the most value when the corpus mixes heterogeneous document types
(legal text + podcast), queries are complex or phrased differently from document language,
and precision matters more than latency.
Baseline + metadata filtering may be sufficient for narrow, homogeneous corpora.

In [ ]:
print('=' * 65)
print('FINAL EVALUATION SUMMARY')
print('=' * 65)
print(df_metrics.to_string())

# Identify best-performing pipeline per metric
for metric in ['precision_at_5', 'mrr', 'ndcg_at_5']:
    if metric in df_metrics.columns:
        best = df_metrics[metric].idxmax()
        val  = df_metrics[metric].max()
        print(f'\nBest {metric:15s}: {best} ({val:.3f})')

In [ ]:
# Quick latency comparison on a single query
if index is None or not HAS_OPENAI:
    print('Skipping latency comparison because OpenAI and Pinecone are both required.')
else:
    q  = TEST_QUERIES[0]
    qe = embed_query(q)
    candidates = retrieve(qe, top_k=INITIAL_TOP_K)

    timings = {}

    t0 = time.time(); candidates[:FINAL_TOP_K]; timings['baseline']     = time.time() - t0
    t0 = time.time(); cohere_rerank(q, candidates, top_n=FINAL_TOP_K);  timings['cohere']       = time.time() - t0
    t0 = time.time(); llm_score_relevance(q, candidates, oai, top_n=FINAL_TOP_K); timings['llm'] = time.time() - t0
    t0 = time.time(); crossencoder_rerank(q, candidates, top_n=FINAL_TOP_K);      timings['crossencoder'] = time.time() - t0

    df_latency = pd.DataFrame([{'pipeline': k, 'latency_s': round(v, 2)} for k, v in timings.items()])
    df_latency = df_latency.set_index('pipeline')
    print('\nLatency (single query):')
    print(df_latency.to_string())
